"""
AI-Generated Text Detection — Data Mining Course Project
==========================================================
Problem: Classify a given piece of text as HUMAN-written or AI-generated (GPT-3.5).

Dataset: "Human-AI-Generated Text Corpus" (LorenzM97, GitHub, public dataset)
          https://github.com/LorenzM97/human-AI-generatedTextCorpus
Subset used here: English-language texts only, balanced 200 Human / 200 AI
          (100 news articles + 100 Wikipedia articles per class).

Pipeline: Data Cleaning -> Feature Engineering (TF-IDF + linguistic features)
          -> Classification (Naive Bayes, Logistic Regression, SVM)
          -> Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
"""


In [1]:
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42

In [2]:
# 1. LOAD DATA
df = pd.read_csv("ai_vs_human_text.csv")
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(df["label"].value_counts())

Dataset shape: (400, 4)
label
AI       200
Human    200
Name: count, dtype: int64


In [3]:
# 2. PREPROCESSING / DATA CLEANING
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.digits))
    text = text.strip()
    return text

df = df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
df["clean_text"] = df["text"].apply(clean_text)
df["label_bin"] = df["label"].map({"Human": 0, "AI": 1})

In [4]:
# 3. FEATURE ENGINEERING
def word_count(t):
    return len(t.split())

def avg_word_len(t):
    words = t.split()
    return np.mean([len(w) for w in words]) if words else 0

def type_token_ratio(t):
    words = t.split()
    return len(set(words)) / len(words) if words else 0

df["word_count"] = df["clean_text"].apply(word_count)
df["avg_word_len"] = df["clean_text"].apply(avg_word_len)
df["lexical_diversity"] = df["clean_text"].apply(type_token_ratio)

print("\nLinguistic feature averages by class:")
print(df.groupby("label")[["word_count", "avg_word_len", "lexical_diversity"]].mean())

vectorizer = TfidfVectorizer(
    max_features=3000, stop_words="english", ngram_range=(1, 2), min_df=2,
)
X = vectorizer.fit_transform(df["clean_text"])
y = df["label_bin"].values


Linguistic feature averages by class:
       word_count  avg_word_len  lexical_diversity
label                                             
AI        311.295      5.184700           0.559402
Human     419.275      5.169461           0.573076


In [5]:
# 4. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


Train size: 320, Test size: 80


In [6]:
# 5. MODELS
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE),
}

results = []
best_model_name, best_f1, best_model, best_preds = None, -1, None, None

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring="f1")
    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1-score": f1, "CV F1 (5-fold mean)": cv_scores.mean()
    })
    if f1 > best_f1:
        best_f1, best_model_name, best_model, best_preds = f1, name, model, preds

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
print("\n=== Model comparison ===")
print(results_df.to_string(index=False))
results_df.to_csv("model_comparison_results.csv", index=False)


=== Model comparison ===
              Model  Accuracy  Precision  Recall  F1-score  CV F1 (5-fold mean)
         Linear SVM    0.6625   0.658537   0.675  0.666667             0.686401
Logistic Regression    0.5875   0.577778   0.650  0.611765             0.558121
        Naive Bayes    0.5250   0.522727   0.575  0.547619             0.476534


In [7]:
# 6. DETAILED EVALUATION OF BEST MODEL
print(f"\n=== Best model: {best_model_name} ===")
print(classification_report(y_test, best_preds, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Human", "AI"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Human", "AI"])
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title(f"Confusion Matrix — {best_model_name}")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
fig.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(results_df))
width = 0.2
for i, metric in enumerate(["Accuracy", "Precision", "Recall", "F1-score"]):
    ax.bar(x + i * width, results_df[metric], width, label=metric)
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(results_df["Model"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — AI vs Human Text Detection")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.close()

lr = models["Logistic Regression"]
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = lr.coef_[0]
top_ai = feature_names[np.argsort(coefs)[-15:]][::-1]
top_human = feature_names[np.argsort(coefs)[:15]]

print("\nTop terms associated with AI-generated text:", list(top_ai))
print("Top terms associated with Human-written text:", list(top_human))

with open("top_terms.txt", "w") as f:
    f.write("Top terms indicating AI-generated text:\n")
    f.write(", ".join(top_ai) + "\n\n")
    f.write("Top terms indicating Human-written text:\n")
    f.write(", ".join(top_human) + "\n")

print("\nDone. Outputs saved: model_comparison_results.csv, confusion_matrix.png, "
      "model_comparison.png, top_terms.txt")


=== Best model: Linear SVM ===
              precision    recall  f1-score   support

       Human       0.67      0.65      0.66        40
          AI       0.66      0.68      0.67        40

    accuracy                           0.66        80
   macro avg       0.66      0.66      0.66        80
weighted avg       0.66      0.66      0.66        80


Top terms associated with AI-generated text: ['fans', 'despite', 'history', 'continue', 'violence', 'significant', 'incident', 'unique', 'impact', 'country', 'government', 'addition', 'need', 'potential', 'challenges']
Top terms associated with Human-written text: ['said', 'million', 'says', 'told', 'th', 'called', 'late', 'usually', 'state', 'week', 'database', 'soil', 'century', 'man', 'second']

Done. Outputs saved: model_comparison_results.csv, confusion_matrix.png, model_comparison.png, top_terms.txt
